## Import all necessary libraries

In [ ]:
import os
import sys
import shutil
import random
import glob
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image
from sklearn.model_selection import train_test_split
import yaml
import torch
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')


PyTorch version: 2.7.1
CUDA available: False
Using CPU for training


## Install YOLOv5 via pip

In [ ]:
# Setup YOLOv5 environment
print("📦 Setting up YOLOv5 environment...")

# Install YOLOv5 via pip
try:
    import yolov5
    print("✅ YOLOv5 already available")
except ImportError:
    print("Installing YOLOv5...")
    os.system('pip install yolov5')

# Alternative: Clone YOLOv5 repository for training
if not os.path.exists('yolov5'):
    print("📥 Cloning YOLOv5 repository for training...")
    os.system('git clone https://github.com/ultralytics/yolov5.git')
    print("✅ YOLOv5 repository cloned")

# Install additional requirements
os.system('pip install wandb tensorboard')
print("✅ YOLOv5 environment ready!")


In [ ]:
def prepare_yolov5_dataset(data_dir="data", output_dir="yolo_dataset"):
    # Create directory structure
    for split in ['train', 'val', 'test']:
        os.makedirs(f"{output_dir}/{split}/images", exist_ok=True)
        os.makedirs(f"{output_dir}/{split}/labels", exist_ok=True)
    
    # Collect all image-label pairs from country directories
    all_pairs = []
    country_dirs = [d for d in os.listdir(data_dir) if d.startswith('country_')]
    
    for country_dir in country_dirs:
        images_path = f"{data_dir}/{country_dir}/images"
        labels_path = f"{data_dir}/{country_dir}/labels"
        
        for img_file in glob(f"{images_path}/*.jpg") + glob(f"{images_path}/*.png"):
            base_name = os.path.splitext(os.path.basename(img_file))[0]
            label_file = f"{labels_path}/{base_name}.txt"
            if os.path.exists(label_file):
                all_pairs.append((img_file, label_file))
    
    # Split dataset (70% train, 15% val, 15% test)
    